# EDA 6 - Occupational-ascent comparison (Adopt methodology from Dr Duncan Smith's profession analysis work)

There is no public repo of Duncan's work, we just adopt his method.

Used 2011 and 2021 census occupation data from NOMIS.


In [ ]:
# ── imports ──────────────────────────────────────────
import sys
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})

In [ ]:
# ── paths ──────────────────────────────────────────
ROOT       = here()
sys.path.insert(0, str(ROOT))
DATA_DIR   = ROOT / 'data'       
OUT_DIR    = ROOT / 'outputs'
GEO_PATH   = DATA_DIR / 'london_msoa_2011.geojson'
FIG_DIR    = OUT_DIR / 'comparison_figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

OCC_2011   = DATA_DIR / 'census_occupation_2011_msoa.csv'
OCC_2021   = DATA_DIR / 'census_occupation_2021_msoa.csv'
EDA4       = OUT_DIR  / 'eda4_results_for_phase3_20260626.csv'
ENRICHED   = OUT_DIR  / 'msoa_cascade_features_enriched_20260625.csv'
MSOA_LOOKUP = DATA_DIR / 'MSOA_2011_to_2021_lookup_for_identification.csv'

In [ ]:
from geo_harmonise import build_harmonisation
from map_utils import load_london_msoa, plot_london_categorical, plot_london_choropleth

FRAMES = {'A': 'London flows · London ladder',
          'B': 'London flows · national ladder',
          'C': 'London+external · national ladder'}
TOP_TERCILE = 0.66          # ascent threshold for the divergence maps


In [ ]:
# ── parse a NOMIS occupation bulk file ─────────────────────────────
def parse_nomis_occupation(path):
    """[msoa_cd, total_occ, aff_count]; affluent = SOC major groups 1+2+3.
    Keeps the all-persons block in the 2011 'by sex' file (max total per MSOA)."""
    with open(path, encoding='utf-8', errors='replace') as fh:
        lines = fh.readlines()
    hdr = next(i for i, l in enumerate(lines) if 'super output area' in l.lower())
    df = pd.read_csv(path, skiprows=hdr, header=0).dropna(how='all')
    c0 = df.columns[0]
    df = df[df[c0].astype(str).str.contains('E0', na=False)].copy()
    df['msoa_cd'] = df[c0].astype(str).str.split(':').str[0].str.strip()
    total = df.columns[1]
    grp = {n: next(c for c in df.columns if c.strip().startswith(n)) for n in ['1.', '2.', '3.']}
    for c in [total, *grp.values()]:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.sort_values(total, ascending=False).drop_duplicates('msoa_cd', keep='first')
    df['aff_count'] = df[grp['1.']] + df[grp['2.']] + df[grp['3.']]
    return df[['msoa_cd', total, 'aff_count']].rename(columns={total: 'total_occ'}).reset_index(drop=True)

### variable explanation:

- Indicator of professional: 
    - **top three groups of Managers (1), Professionals (2), and Associate Professionals (3)**. 
    - Residents in these three the "most affluent occupational classes" 
    - Duncan foud the share of these proportions dramatically across East and South-East London.
    - He also stressed that ***this occupational data can't tell whether gentrification processes displace low income people in London***. Specifically, these are demographic data with residential composition measure.

### Limitation in data:

- 2011 and 2021 use slightly different population bases (2011 = residents aged 16–74 in employment; 2021 = aged 16 and over in employment). 
- 2011 and 2021 use different classifications: 2011 used SOC2010, 2021 used SOC2020.
    - people coded to SOC2010 Major Group 3 (Associate Professional and Technical), about 11% were recoded into Major Group 2 (Professional) under SOC2020, with 88% staying in Group 3.
    - some of the apparent "rise in professionals" between 2011 and 2021 is a coding artefact, not real social change.
    - BUT, 
        - The dominant SOC2010→SOC2020 reclassification moves people *between* groups 3 and 2, we have aggregated both. 
        - Those moves cancel within the bundle.

In [ ]:
# ── build occupational ascent (geo_harmonise → full 982 coverage) ──
e4 = pd.read_csv(EDA4)
frame = e4['msoa11cd'].tolist()

# Build the 2021→2011 remap directly from geo_harmonise
h = build_harmonisation(pd.read_csv(MSOA_LOOKUP), frame)
print(f'geo_harmonise: collapse={h.collapse_2011}, frame={len(h.frame)} MSOAs')

o11 = parse_nomis_occupation(OCC_2011).rename(columns={'msoa_cd': 'msoa11cd'})
o11['aff_share_11'] = o11['aff_count'] / o11['total_occ']

o21 = parse_nomis_occupation(OCC_2021)
o21['msoa11cd'] = o21['msoa_cd'].map(h.remap_2021).fillna(o21['msoa_cd'])   # 2021→2011
o21 = o21.groupby('msoa11cd', as_index=False)[['total_occ', 'aff_count']].sum()  # sum splits/merges
o21['aff_share_21'] = o21['aff_count'] / o21['total_occ']

df = (e4.merge(o11[['msoa11cd', 'aff_share_11']], on='msoa11cd', how='left')
        .merge(o21[['msoa11cd', 'aff_share_21']], on='msoa11cd', how='left'))
enr = pd.read_csv(ENRICHED)
df = df.merge(enr[['msoa11cd', 'Net_Cascade_11', 'Net_Cascade_21']], on='msoa11cd', how='left')
df['ascent'] = df['aff_share_21'] - df['aff_share_11']

cov = df['ascent'].notna().sum()
print(f'Occupational ascent built for {cov}/{len(df)} MSOAs')
print(f"  mean={df['ascent'].mean():+.4f}  median={df['ascent'].median():+.4f}  "
      f"%positive={(df['ascent']>0).mean()*100:.1f}%  (near-universal → compressed variance)")


### variable explantion

"ascent" means increase in shares of top-three occupational residents.

Because of data limitation mentioned above, we should treat `ascent` as a *coarse, broadly-comparable* indicator

### Interpretation

***Occupational rise is near-universal (96% ascent) in London.***
- Most MSOAs are positive, since the professionalisation affects the whole economy.
- The limitation of **pure occupational analysis is it can't ditinguish the displacement-driven gentrification from a city-wide trend**.
    - I feel this was most confusion part for me before, I though gentrification was a city-wide trend, but results told me it's a very localised phenomenon, as minority!

In [ ]:
# ── cross-frame discriminant test (A / B / C, 2021 leg) ────────────
print('Occupational ascent vs 2021 flow typology, by frame')
print('=' * 60)
rhos = {}
for X, name in FRAMES.items():
    r, p = stats.spearmanr(df['ascent'], df[f'Dom_{X}_21'], nan_policy='omit')
    rhos[X] = r
    print(f'\nFrame {X} ({name})')
    print(f'  ρ(ascent, Dom_{X}_21) = {r:+.3f}  (p={p:.1e})')
    g = df.groupby(f'Typ_{X}_21')['ascent'].agg(['mean', 'count'])
    for t, row in g.iterrows():
        print(f'     {str(t):12s} mean={row["mean"]:+.4f}  n={int(row["count"])}')
    grps = [gg['ascent'].dropna().values for _, gg in df.groupby(f'Typ_{X}_21') if len(gg) > 1]
    H, pk = stats.kruskal(*grps)
    print(f'     Kruskal-Wallis H={H:.1f}  p={pk:.1e}')

print('\nFRAME SENSITIVITY (2021):  ' +
      '   '.join(f'{X}: ρ={rhos[X]:+.3f}' for X in FRAMES))
print('London ladder (A) is mildly negative (decile-slice artefact); the')
print('national+external frame (C) is ~0 → ascent and cascade are independent.')


### Interpretaion:

Bottom results confirmed that we should use Frame C, and the artefacts of London-only ladder did affect the comparison results.

***Occupational ascent are independent from cascade typology***.

In [ ]:
# ── divergence categories (flow vs occupational attribute) ─────────
FLOW_TYP = 'Typ_C_21'          # main analysis frame
thr = df['ascent'].quantile(TOP_TERCILE)

def occ_divergence(row):
    hi = row['ascent'] >= thr
    c = row[FLOW_TYP]
    if hi and c == 'Counter-led':  return 'occ ascent but counter-led'   # research gap
    if hi and c == 'Cascade-led':  return 'occ ascent & cascade'
    if (not hi) and c == 'Cascade-led': return 'cascade, occ flat'
    return 'other'

df['OccDiv'] = df.apply(occ_divergence, axis=1)
print(df['OccDiv'].value_counts().to_string())

OCC_DIV_COLORS = {
    'occ ascent but counter-led':   '#d73027',   # red  — where attribute methods mislead
    'cascade, occ flat': '#4575b4',   # blue — where flow adds detection
    'occ ascent & cascade':        '#1a9850',   # green — convergence
    'other':              '#e0e0e0',   # grey
}


In [ ]:
# ── MAP 1 · occupation–flow divergence (the research gap) ──────────
gdf = load_london_msoa(GEO_PATH, df)
fig, ax = plt.subplots(figsize=(11, 9))
plot_london_categorical(
    gdf, column='OccDiv',
    title='Where flow and occupational-ascent methods disagree (2021)',
    color_dict=OCC_DIV_COLORS, ax=ax)
if ax.get_legend():
    ax.get_legend().remove()
fig.legend(handles=[mpatches.Patch(color=c, label=l) for l, c in OCC_DIV_COLORS.items()],
           loc='lower center', bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig03_map_occupation_divergence_2021.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── MAP 2 · occupational ascent choropleth (context: near-universal) ─
gdf = load_london_msoa(GEO_PATH, df)
fig, ax = plot_london_choropleth(
    gdf, column='ascent',
    title='Occupational ascent 2011→2021 (delta share in SOC groups 1–3)',
    cmap='PuBuGn',
    legend_label='delta affluent occupational share')
plt.savefig(FIG_DIR / 'fig04_map_occupation_ascent.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── export ─────────────────────────────────────────────────────────
out = OUT_DIR / f'occupation_comparison_20260627.csv'
df.to_csv(out, index=False)
print(f'✓ saved {out}  ({df.shape[0]} x {df.shape[1]})')
